In [33]:
import duckdb
import pandas as pd

con = duckdb.connect()

path = "../../source_database/seed_data/rewardsco_loyalty_transactions.csv"
stores_path = "../../source_database/seed_data/stores.csv"

In [9]:
# Schema

# WHY: standard first pass. NOTE before you read the null %s: basket_value_gbp
# and sku are EXPECTED to be null for every 'redeem' row (you don't buy a
# basket when you redeem points) — so don't treat their null rate as a bug
# until Cell 4 separates it out by transaction_type.

schema = con.sql(f"DESCRIBE SELECT * FROM read_csv_auto('{path}')").df()
print("=== SCHEMA ===")
print(schema)

=== SCHEMA ===
        column_name column_type null   key default extra
0    transaction_id      BIGINT  YES  None    None  None
1         member_id      BIGINT  YES  None    None  None
2  transaction_type     VARCHAR  YES  None    None  None
3            points      BIGINT  YES  None    None  None
4  basket_value_gbp      DOUBLE  YES  None    None  None
5          store_id      BIGINT  YES  None    None  None
6           channel     VARCHAR  YES  None    None  None
7               sku     VARCHAR  YES  None    None  None
8    transaction_at   TIMESTAMP  YES  None    None  None
9        created_at   TIMESTAMP  YES  None    None  None


In [10]:
# ── Completeness Audit: how many NULLs does each column have? ──

# Step 1: Get the list of column names automatically from the file.
# This way the same audit code works on ANY csv
columns_df = con.sql(f"DESCRIBE SELECT * FROM read_csv_auto('{path}')").df()
column_names = columns_df['column_name'].tolist()

# Step 2: Get the total row count once, so we can turn null counts into %.
total_rows = con.sql(f"SELECT COUNT(*) AS cnt FROM read_csv_auto('{path}')").df()['cnt'][0]
print(f"Total rows in file: {total_rows}")

# Step 3: For each column, ask DuckDB:
#   COUNT(*)        -> counts every row (NULL or not)
#   COUNT(col_name) -> counts only NON-null rows for that column
#   so COUNT(*) - COUNT(col_name) = how many rows are NULL in that column
null_report = []
for col in column_names:
    result = con.sql(f"""
        SELECT
            '{col}' AS column_name,
            COUNT(*) - COUNT({col}) AS null_count,
            ROUND(100.0 * (COUNT(*) - COUNT({col})) / COUNT(*), 2) AS null_percentage
        FROM read_csv_auto('{path}')
    """).df()
    null_report.append(result)

# Step 4: Stack all the per-column results into one table and show it.
import pandas as pd
null_report_df = pd.concat(null_report, ignore_index=True)
null_report_df

Total rows in file: 13439


,column_name,null_count,null_percentage
0,transaction_id,0,0.00
1,member_id,0,0.00
2,transaction_type,0,0.00
3,points,0,0.00
4,basket_value_gbp,4432,32.98
5,store_id,2733,20.34
6,channel,0,0.00
7,sku,4432,32.98
8,transaction_at,0,0.00
9,created_at,0,0.00


In [34]:
# ──redeem rows that unexpectedly HAVE a basket_value_gbp or sku ──
# WHY: a 'redeem' event shouldn't carry a basket/product — if either is NOT null here,
# that row's transaction_type may be mis-tagged, or the row is otherwise corrupted.
redeem_with_values = con.sql(f"""
    SELECT *
    FROM read_csv_auto('{path}')
    WHERE transaction_type = 'redeem'
      AND basket_value_gbp IS NOT NULL
      AND sku IS NOT NULL
""").df()
print(f"'redeem' rows that have BOTH basket_value_gbp and sku populated: {len(redeem_with_values)}")
redeem_with_values

'redeem' rows that have BOTH basket_value_gbp and sku populated: 0


,transaction_id,member_id,transaction_type,points,basket_value_gbp,store_id,channel,sku,transaction_at,created_at


In [35]:
# ──non-redeem (i.e. 'earn') rows that are MISSING basket_value_gbp or sku ──
# WHY: an 'earn' event always comes from a real purchase — it should always carry a
# basket value and a sku. If either is null here, that row is incomplete.
non_redeem_missing_values = con.sql(f"""
    SELECT *
    FROM read_csv_auto('{path}')
    WHERE transaction_type != 'redeem'
      AND (basket_value_gbp IS NULL OR sku IS NULL)
""").df()
print(f"Non-'redeem' rows missing basket_value_gbp or sku: {len(non_redeem_missing_values)}")
non_redeem_missing_values

Non-'redeem' rows missing basket_value_gbp or sku: 0


,transaction_id,member_id,transaction_type,points,basket_value_gbp,store_id,channel,sku,transaction_at,created_at


In [11]:
# Uniqueness

# WHY: transaction_id is the primary key for an event log — a duplicate
# means the same loyalty event got counted twice, which directly inflates
# member point balances. Also checking for an exact duplicate EVENT (same
# member, same timestamp, same points, same sku) — a stricter check that
# catches a re-sent/replayed event even if it got a new transaction_id.

dupe_pk = con.sql(f"""
    SELECT transaction_id, COUNT(*) AS n
    FROM read_csv_auto('{path}')
    GROUP BY transaction_id HAVING COUNT(*) > 1
""").df()
print(f"Duplicate transaction_id: {len(dupe_pk)}")
print(dupe_pk)

dupe_event = con.sql(f"""
    SELECT member_id, transaction_at, points, sku, COUNT(*) AS n
    FROM read_csv_auto('{path}')
    GROUP BY member_id, transaction_at, points, sku
    HAVING COUNT(*) > 1
""").df()
print(f"\nExact duplicate events (same member/time/points/sku): {len(dupe_event)}")
print(dupe_event.head(20))

Duplicate transaction_id: 0
Empty DataFrame
Columns: [transaction_id, n]
Index: []

Exact duplicate events (same member/time/points/sku): 0
Empty DataFrame
Columns: [member_id, transaction_at, points, sku, n]
Index: []


In [12]:
# Referential integrity: store_id

# WHY: store_id should only ever point to a real store. member_id would
# ideally be checked against a loyalty-members table too, but that file
# isn't among your uploads yet — this is written defensively so it tells
# you that plainly instead of silently skipping it.

orphan_stores = con.sql(f"""
    SELECT COUNT(*) FROM read_csv_auto('{path}') t
    LEFT JOIN read_csv_auto('{stores_path}') s ON t.store_id = s.store_id
    WHERE t.store_id IS NOT NULL AND s.store_id IS NULL
""").fetchone()[0]
print(f"store_id values with no match in stores.csv (excluding nulls): {orphan_stores}")

print("\nNo members.csv / loyalty_members.csv found in this audit — "
      "member_id is NOT checked against a parent table. Add it here if that file exists.")

store_id values with no match in stores.csv (excluding nulls): 0

No members.csv / loyalty_members.csv found in this audit — member_id is NOT checked against a parent table. Add it here if that file exists.


In [13]:
# channel vs store_id: the headline finding

# WHY: an in_store (and arguably click_collect) transaction physically
# happened at a store — store_id shouldn't be null for those. online is the
# one channel where "no store" is legitimate. This breaks the null rate down
# BY channel so you can see exactly where the gap is instead of one vague
# completeness percentage.

store_id_by_channel = con.sql(f"""
    SELECT channel,
           COUNT(*) AS total,
           COUNT(*) - COUNT(store_id) AS null_store_id,
           ROUND(100.0 * (COUNT(*) - COUNT(store_id)) / COUNT(*), 1) AS pct_null
    FROM read_csv_auto('{path}')
    GROUP BY channel
    ORDER BY pct_null DESC
""").df()
print("=== store_id null rate by channel ===")
print(store_id_by_channel)

# The clearest violation: in_store transactions with no store recorded.
in_store_missing_store = con.sql(f"""
    SELECT * FROM read_csv_auto('{path}')
    WHERE channel = 'in_store' AND store_id IS NULL
""").df()
print(f"\nin_store transactions with NULL store_id (should be 0): {len(in_store_missing_store)}")
print(in_store_missing_store.head(10))

=== store_id null rate by channel ===
         channel  total  null_store_id  pct_null
0         online   4469            939      21.0
1  click_collect   4469            896      20.0
2       in_store   4501            898      20.0

in_store transactions with NULL store_id (should be 0): 898
   transaction_id  member_id transaction_type  points  basket_value_gbp  \
0            1857        412             earn     134             81.05   
1            8388        207           redeem     280               NaN   
2           11405        358           redeem     219               NaN   
3            2922        321             earn     117            107.18   
4           10255        174           redeem     227               NaN   
5             344         30           redeem     240               NaN   
6           13138        269             earn     190            176.74   
7            8429        434             earn     438             89.65   
8            6180        140  

In [14]:
# transaction_type must be a clean, known value

# WHY: this column drives sign logic for the whole table (see Cell 4d) —
# a typo'd or unexpected value here silently breaks any balance calculation
# built on top of it.

print(con.sql(f"""
    SELECT transaction_type, COUNT(*) AS n
    FROM read_csv_auto('{path}')
    GROUP BY transaction_type
    ORDER BY n DESC
""").df())

  transaction_type     n
0             earn  9007
1           redeem  4432


In [15]:
# points must be positive, and basket_value_gbp must be positive when present

# WHY: points is stored as magnitude-only (confirmed: zero negative values
# in the file), so a value of 0 or negative is invalid regardless of
# transaction_type. basket_value_gbp should never be zero/negative when it's
# actually populated (only 'earn' rows should have it — checked in 4e).

bad_points = con.sql(f"SELECT * FROM read_csv_auto('{path}') WHERE points <= 0").df()
print(f"points <= 0: {len(bad_points)}")
print(bad_points.head(10))

bad_basket = con.sql(f"""
    SELECT * FROM read_csv_auto('{path}')
    WHERE basket_value_gbp IS NOT NULL AND basket_value_gbp <= 0
""").df()
print(f"\nbasket_value_gbp <= 0 (where present): {len(bad_basket)}")
print(bad_basket.head(10))

points <= 0: 0
Empty DataFrame
Columns: [transaction_id, member_id, transaction_type, points, basket_value_gbp, store_id, channel, sku, transaction_at, created_at]
Index: []

basket_value_gbp <= 0 (where present): 0
Empty DataFrame
Columns: [transaction_id, member_id, transaction_type, points, basket_value_gbp, store_id, channel, sku, transaction_at, created_at]
Index: []


In [16]:
# the sign-convention trap: does a naive SUM(points) give the wrong balance?

# WHY: this is the most important check in the table. points has no sign —
# 'redeem' rows store a positive number even though they should SUBTRACT
# from a member's balance. This cell shows, in concrete numbers, how wrong
# a naive balance calculation would be if someone just does SUM(points)
# instead of applying transaction_type as the sign.

balance_comparison = con.sql(f"""
    SELECT
        SUM(points) AS naive_sum_wrong,
        SUM(CASE WHEN transaction_type = 'redeem' THEN -points ELSE points END) AS correct_net_balance
    FROM read_csv_auto('{path}')
""").df()
print("=== Naive SUM(points) vs correct signed balance, across the whole table ===")
print(balance_comparison)
print(f"\nDifference: {balance_comparison['naive_sum_wrong'][0] - balance_comparison['correct_net_balance'][0]:,} "
      f"points of overstatement if signs aren't applied.")

=== Naive SUM(points) vs correct signed balance, across the whole table ===
   naive_sum_wrong  correct_net_balance
0        3062372.0            1524034.0

Difference: 1,538,338.0 points of overstatement if signs aren't applied.


In [17]:
# channel/category check: 'earn' always has basket_value_gbp + sku, 'redeem' never does

# WHY: these two fields should move together with transaction_type. A
# 'redeem' row WITH a basket_value_gbp or sku, or an 'earn' row WITHOUT
# them, means either the transaction_type was mis-tagged or the rest of
# the row is corrupted.

inconsistent_earn = con.sql(f"""
    SELECT * FROM read_csv_auto('{path}')
    WHERE transaction_type = 'earn' AND (basket_value_gbp IS NULL OR sku IS NULL)
""").df()
print(f"'earn' rows missing basket_value_gbp or sku: {len(inconsistent_earn)}")
print(inconsistent_earn.head(10))

inconsistent_redeem = con.sql(f"""
    SELECT * FROM read_csv_auto('{path}')
    WHERE transaction_type = 'redeem' AND (basket_value_gbp IS NOT NULL OR sku IS NOT NULL)
""").df()
print(f"\n'redeem' rows that unexpectedly HAVE basket_value_gbp or sku: {len(inconsistent_redeem)}")
print(inconsistent_redeem.head(10))

'earn' rows missing basket_value_gbp or sku: 0
Empty DataFrame
Columns: [transaction_id, member_id, transaction_type, points, basket_value_gbp, store_id, channel, sku, transaction_at, created_at]
Index: []

'redeem' rows that unexpectedly HAVE basket_value_gbp or sku: 0
Empty DataFrame
Columns: [transaction_id, member_id, transaction_type, points, basket_value_gbp, store_id, channel, sku, transaction_at, created_at]
Index: []


In [18]:
# sku format consistency

# WHY: sku should always follow the NP-SKU-##### pattern when present
# (i.e. for 'earn' rows) — a malformed sku breaks any join back to a
# products table by sku.

bad_sku = con.sql(f"""
    SELECT * FROM read_csv_auto('{path}')
    WHERE sku IS NOT NULL AND sku !~ '^NP-SKU-\\d{{5}}$'
""").df()
print(f"Malformed sku (where present): {len(bad_sku)}")
print(bad_sku.head(10))

Malformed sku (where present): 0
Empty DataFrame
Columns: [transaction_id, member_id, transaction_type, points, basket_value_gbp, store_id, channel, sku, transaction_at, created_at]
Index: []


In [20]:
# distribution / outliers

# WHY: catches entry errors that pass the basic validity check (e.g. a
# basket_value_gbp of £5,000 is "valid" but almost certainly wrong for a
# retail loyalty transaction).

def iqr_outliers(con, path, col):
    return con.sql(f"""
        WITH filtered AS (
            SELECT transaction_id, {col}
            FROM read_csv_auto('{path}')
            WHERE {col} IS NOT NULL
        ),
        stats AS (
            SELECT QUANTILE_CONT({col}, 0.25) AS q1,
                   QUANTILE_CONT({col}, 0.75) AS q3
            FROM filtered
        )
        SELECT f.transaction_id, f.{col}
        FROM filtered f, stats
        WHERE f.{col} < (stats.q1 - 1.5*(stats.q3-stats.q1))
           OR f.{col} > (stats.q3 + 1.5*(stats.q3-stats.q1))
        ORDER BY f.{col} DESC
    """).df()

print("=== Outliers: points ===")
print(iqr_outliers(con, path, "points").head(15))

print("\n=== Outliers: basket_value_gbp ===")
print(iqr_outliers(con, path, "basket_value_gbp").head(15))

=== Outliers: points ===
Empty DataFrame
Columns: [transaction_id, points]
Index: []

=== Outliers: basket_value_gbp ===
Empty DataFrame
Columns: [transaction_id, basket_value_gbp]
Index: []


In [21]:
# freshness / volume (and the 3-month gap)

# WHY: this table's latest transaction_at is 2026-06-30 — if today's date
# is well past that, it means the feed has stopped, not that business
# stopped. This cell shows row counts by month so you can see exactly
# where volume drops off, and explicitly reports the gap to "today".

monthly_volume = con.sql(f"""
    SELECT strftime(CAST(transaction_at AS TIMESTAMP), '%Y-%m') AS month,
           COUNT(*) AS n_rows
    FROM read_csv_auto('{path}')
    GROUP BY month
    ORDER BY month
""").df()
print("=== Row volume by month ===")
print(monthly_volume)

latest = con.sql(f"SELECT MAX(transaction_at) AS latest_transaction FROM read_csv_auto('{path}')").df()
print("\n=== Latest transaction in the file ===")
print(latest)
print("Compare this to today's date — a large gap means the pipeline has likely stopped feeding this table.")

=== Row volume by month ===
      month  n_rows
0   2025-01     716
1   2025-02     688
2   2025-03     733
3   2025-04     736
4   2025-05     836
5   2025-06     750
6   2025-07     789
7   2025-08     762
8   2025-09     726
9   2025-10     798
10  2025-11     719
11  2025-12     720
12  2026-01     796
13  2026-02     686
14  2026-03     727
15  2026-04     784
16  2026-05     754
17  2026-06     719

=== Latest transaction in the file ===
   latest_transaction
0 2026-06-30 22:32:00
Compare this to today's date — a large gap means the pipeline has likely stopped feeding this table.


In [24]:
# time drift: transaction_at vs created_at

# WHY: in an event-log table these CAN legitimately differ (a transaction
# that happened at 9am might not be written to the warehouse until 9:05am
# in a real late-arriving pipeline). This checks whether that's actually
# happening here, or whether the two columns are just duplicates of each
# other (which would mean created_at carries no extra information).

drift = con.sql(f"""
    SELECT transaction_id, transaction_at, created_at,
           date_diff('second', transaction_at, created_at) AS lag_seconds
    FROM read_csv_auto('{path}')
    WHERE transaction_at != created_at
""").df()
print(f"Rows where created_at != transaction_at: {len(drift)}")
print(drift.describe() if len(drift) else "created_at is identical to transaction_at for every row.")

print("\n---------------------------------------------------------------------")

drift

Rows where created_at != transaction_at: 20
       transaction_id       transaction_at           created_at  lag_seconds
count        20.00000                   20                   20         20.0
mean      13429.50000  2025-11-03 11:07:31  2025-11-03 11:07:30         -1.0
min       13420.00000  2025-03-30 20:27:01  2025-03-30 20:27:00         -1.0
25%       13424.75000  2025-09-02 18:08:31  2025-09-02 18:08:30         -1.0
50%       13429.50000  2025-10-23 06:02:31  2025-10-23 06:02:30         -1.0
75%       13434.25000  2026-01-16 07:30:46  2026-01-16 07:30:45         -1.0
max       13439.00000  2026-06-27 08:58:01  2026-06-27 08:58:00         -1.0
std           5.91608                  NaN                  NaN          0.0

---------------------------------------------------------------------


,transaction_id,transaction_at,created_at,lag_seconds
0,13427,2025-03-30 20:27:01,2025-03-30 20:27:00,-1
1,13422,2025-05-15 18:27:01,2025-05-15 18:27:00,-1
2,13429,2025-07-03 09:12:01,2025-07-03 09:12:00,-1
3,13425,2025-07-07 20:12:01,2025-07-07 20:12:00,-1
4,13428,2025-09-02 12:01:01,2025-09-02 12:01:00,-1
5,13424,2025-09-02 20:11:01,2025-09-02 20:11:00,-1
6,13433,2025-09-10 14:01:01,2025-09-10 14:01:00,-1
7,13437,2025-09-11 17:00:01,2025-09-11 17:00:00,-1
8,13436,2025-10-09 16:07:01,2025-10-09 16:07:00,-1
9,13426,2025-10-21 14:54:01,2025-10-21 14:54:00,-1


In [32]:
# ── Format Consistency Audit: same value, inconsistent spelling/casing? ──

text_columns = ['channel', 'transaction_type']

consistency_report = []
for col in text_columns:
    result = con.sql(f"""
        SELECT
            '{col}' AS column_name,
            COUNT(DISTINCT {col}) AS raw_distinct_count,
            COUNT(DISTINCT LOWER(TRIM({col}))) AS normalized_distinct_count
        FROM read_csv_auto('{path}')
    """).df()
    consistency_report.append(result)

import pandas as pd
consistency_df = pd.concat(consistency_report, ignore_index=True)

# If raw > normalized, some values are duplicates of each other under
# different casing/spacing — this column shows exactly how many.
consistency_df['extra_variants'] = (
    consistency_df['raw_distinct_count'] - consistency_df['normalized_distinct_count']
)
consistency_df


,column_name,raw_distinct_count,normalized_distinct_count,extra_variants
0,channel,3,3,0
1,transaction_type,2,2,0


In [31]:
# For any column above with extra_variants > 0, list its actual raw values
# so you can see the exact inconsistency (e.g. "Online" vs "online" vs " Online").
for col in text_columns:
    values_df = con.sql(f"""
        SELECT DISTINCT {col}
        FROM read_csv_auto('{path}')
        ORDER BY {col}
    """).df()
    print(f"\n{col} — {len(values_df)} distinct raw value(s):")
    print(values_df[col].tolist())


channel — 3 distinct raw value(s):
['click_collect', 'in_store', 'online']

transaction_type — 2 distinct raw value(s):
['earn', 'redeem']


In [25]:
# exploratory: does points scale with basket_value_gbp?

# WHY: not a hard rule (I don't have a confirmed "£1 = N points" formula
# the way order_items had a confirmed line_total formula) — this is
# EXPLORATORY. It computes points-per-pound for every 'earn' row so you can
# see by eye whether there's a consistent rate, a few distinct tiers, or
# genuinely random values. Treat the output as a lead to investigate, not
# a pass/fail check.

rate_check = con.sql(f"""
    SELECT transaction_id, points, basket_value_gbp,
           ROUND(points / basket_value_gbp, 2) AS points_per_pound
    FROM read_csv_auto('{path}')
    WHERE transaction_type = 'earn' AND basket_value_gbp > 0
""").df()

print(rate_check['points_per_pound'].describe())
print("\nIf this clusters tightly around one or two numbers, that's your earn-rate formula.")
print("If it's all over the place, points are likely randomly generated / promo-driven, not formula-based.")

count    9007.000000
mean        5.179326
std         8.139250
min         0.060000
25%         1.405000
50%         2.730000
75%         5.110000
max        89.350000
Name: points_per_pound, dtype: float64

If this clusters tightly around one or two numbers, that's your earn-rate formula.
If it's all over the place, points are likely randomly generated / promo-driven, not formula-based.


In [26]:
# Format consistency: Flag columns that contain mixed data types
def format_consistency_check(con, path):
    # Step 1: get every column name in the file
    schema = con.sql(f"DESCRIBE SELECT * FROM read_csv_auto('{path}')").df()
    columns = schema['column_name'].tolist()

    all_rows = []
    for col in columns:
        # Step 2: for THIS column, classify every value into a bucket
        breakdown = con.sql(f"""
            SELECT
                '{col}' AS column_name,
                CASE
                    WHEN "{col}" IS NULL OR TRIM("{col}") = '' THEN 'NULL/EMPTY'
                    WHEN TRY_CAST("{col}" AS BIGINT) IS NOT NULL THEN 'INTEGER'
                    WHEN TRY_CAST("{col}" AS DOUBLE) IS NOT NULL THEN 'DECIMAL'
                    WHEN TRY_CAST("{col}" AS DATE) IS NOT NULL THEN 'DATE'
                    WHEN TRY_CAST("{col}" AS TIMESTAMP) IS NOT NULL THEN 'TIMESTAMP'
                    WHEN lower("{col}") IN ('true','false') THEN 'BOOLEAN'
                    ELSE 'TEXT'
                END AS detected_type,
                COUNT(*) AS row_count
            FROM read_csv_auto('{path}', all_varchar=True)
            GROUP BY detected_type
            ORDER BY row_count DESC
        """).df()
        all_rows.append(breakdown)

    result = pd.concat(all_rows, ignore_index=True)

    # Step 3: flag a column only if it has 2+ real (non-null) types
    n_types = result[result['detected_type'] != 'NULL/EMPTY'].groupby('column_name')['detected_type'].transform('nunique')
    result['is_mixed'] = (n_types.reindex(result.index).fillna(0) > 1)
    return result

report = format_consistency_check(con, path)
report                       # full breakdown — bare, no print(), so it renders as a table

,column_name,detected_type,row_count,is_mixed
0,transaction_id,INTEGER,13439,False
1,member_id,INTEGER,13439,False
2,transaction_type,TEXT,13439,False
3,points,INTEGER,13439,False
4,basket_value_gbp,INTEGER,9007,False
5,basket_value_gbp,NULL/EMPTY,4432,False
6,store_id,INTEGER,10706,False
7,store_id,NULL/EMPTY,2733,False
8,channel,TEXT,13439,False
9,sku,TEXT,9007,False


In [27]:
# Sentinel/Default value usage:

# Part 1: scan for known/common sentinel patterns
SENTINEL_VALUES = [
    '-1', '0', '999', '9999', '99999', '-999',
    'NA', 'N/A', 'NULL', 'NONE', 'UNKNOWN', 'UNK', 'TBD', '-', '?',
    '1900-01-01', '1970-01-01', '9999-12-31', '0001-01-01'
]

def sentinel_scan(con, path, sentinel_values):
    schema = con.sql(f"DESCRIBE SELECT * FROM read_csv_auto('{path}')").df()
    columns = schema['column_name'].tolist()
    values_sql = ", ".join(f"'{v.upper()}'" for v in sentinel_values)

    all_rows = []
    for col in columns:
        df = con.sql(f"""
            SELECT
                '{col}' AS column_name,
                "{col}" AS value,
                COUNT(*) AS row_count
            FROM read_csv_auto('{path}', all_varchar=True)
            WHERE upper(TRIM("{col}")) IN ({values_sql})
            GROUP BY "{col}"
            ORDER BY row_count DESC
        """).df()
        all_rows.append(df)

    return pd.concat(all_rows, ignore_index=True)

sentinel_report = sentinel_scan(con, path, SENTINEL_VALUES)
sentinel_report


,column_name,value,row_count
0,transaction_id,999,1
1,transaction_id,9999,1
